# Export Processed Sessions for data.syn.bike

This utility exports one CSV per selected processed session. Select sessions or persisted aggregations with the session selector, set the output directory, then run the export cell.

The exporter uses the processed session dataframe and signal registry. For front and rear raw channels it looks for raw/count signals by semantic `end`, preferring `domain == "wheel"` over `domain == "suspension"` when both exist. If a raw channel is marked or configured as inverted, the exported value is flipped back as `ADC_MAX_COUNT - value`.

The exported CSV is headerless, as required by data.syn.bike. Columns are emitted in this order: `Sample Time`, `Front Raw`, `Rear Raw`, `Lat`, `Long`, `Speed`.

`Sample Time` is exported as a zero-based sample count, matching the original SynDAQ logger convention used by the local processor.

In [1]:
from pathlib import Path

# ---- Settings -------------------------------------------------------------
ARTIFACTS_DIR = Path("artifacts")
OUTPUT_DIR = Path("exports/data_syn_bike")

# data.syn.bike requires a headerless CSV in this exact column order.
# These labels are used only internally while building the dataframe.
DATA_SYN_BIKE_COLUMNS = {
    "time": "Sample Time",
    "front_raw": "Front Raw",
    "rear_raw": "Rear Raw",
    "lat": "Lat",
    "lon": "Long",
    "speed": "Speed",
}

# BODAQS ADC count range. Inverted raw channels are exported as ADC_MAX_COUNT - n.
ADC_MAX_COUNT = 4095

# Manual inversion overrides for artifacts that do not preserve logger calibration metadata.
# Values here are ORed with any inversion flag found in the signal metadata.
INVERT_RAW_BY_END = {
    "front": False,
    "rear": False,
}

# Optional exact dataframe-column overrides, useful for one-off historical artifacts.
INVERT_RAW_COLUMNS = set()

# SynDAQ/data.syn.bike expects an unsigned-style sample counter in column 0.
# The local SynDAQ Python processor ignores this column and reconstructs time from spec['freq'],
# but browser-side code appears to expect a UInt-shaped value here.
TIME_FORMAT = "sample_count"

# GPS speed in BODAQS/FIT artifacts is normally m/s. Change if data.syn.bike expects another unit.
SPEED_MULTIPLIER = 1.0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifacts: {ARTIFACTS_DIR.resolve()}")
print(f"Output:    {OUTPUT_DIR.resolve()}")

Artifacts: D:\Dev\BODAQS\analysis\artifacts
Output:    D:\Dev\BODAQS\analysis\exports\data_syn_bike


In [2]:
from IPython.display import display
from bodaqs_analysis.widgets.session_selector import make_session_selector

selector = make_session_selector(
    artifacts_dir=ARTIFACTS_DIR,
    select_first_by_default=True,
)
display(selector["ui"])

In [5]:
import math
import re
from typing import Any, Mapping

import numpy as np
import pandas as pd

from bodaqs_analysis.artifacts import ArtifactStore, load_session_artifacts


LAT_COLS = (
    "gps_fit_position_latitude_dom_world [deg]",
    "gps_position_latitude_dom_world [deg]",
    "latitude_deg",
    "lat",
)
LON_COLS = (
    "gps_fit_position_longitude_dom_world [deg]",
    "gps_position_longitude_dom_world [deg]",
    "longitude_deg",
    "lon",
    "long",
)
SPEED_COLS = (
    "gps_fit_enhanced_speed_dom_world [m/s]",
    "gps_fit_speed_dom_world [m/s]",
    "gps_speed_dom_world [m/s]",
    "speed_mps",
    "speed",
)


def _safe_filename_part(value: str) -> str:
    text = str(value or "").strip() or "session"
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", text)


def _signals(session: Mapping[str, Any]) -> dict[str, dict[str, Any]]:
    signals = ((session.get("meta") or {}).get("signals") or {})
    return {str(k): dict(v) for k, v in signals.items() if isinstance(v, Mapping)}


def _norm(value: Any) -> str:
    return str(value or "").strip().lower()


def _is_raw_signal(col: str, info: Mapping[str, Any]) -> bool:
    quantity = _norm(info.get("quantity"))
    kind = _norm(info.get("kind"))
    unit = _norm(info.get("unit"))
    name = _norm(col)
    return quantity == "raw" or kind == "raw" or unit == "counts" or "_raw" in name or "[counts]" in name


def _raw_domain_score(info: Mapping[str, Any], col: str) -> int:
    domain = _norm(info.get("domain"))
    name = _norm(col)
    if domain == "wheel" or "wheel" in name:
        return 0
    if domain == "suspension" or "shock" in name or "fork" in name:
        return 1
    return 2


def _pick_raw_column(session: Mapping[str, Any], end: str) -> tuple[str | None, dict[str, Any], str]:
    df = session["df"]
    end = _norm(end)
    candidates: list[tuple[int, str, dict[str, Any], str]] = []

    for col, info in _signals(session).items():
        if col not in df.columns:
            continue
        if _norm(info.get("end")) != end:
            continue
        if not _is_raw_signal(col, info):
            continue
        candidates.append((_raw_domain_score(info, col), col, info, "signal registry"))

    if not candidates:
        end_tokens = ("front", "fork") if end == "front" else ("rear", "shock")
        for col in df.columns:
            name = _norm(col)
            if not any(tok in name for tok in end_tokens):
                continue
            if "raw" not in name and "counts" not in name:
                continue
            info = {"end": end, "quantity": "raw", "unit": "counts", "domain": None}
            candidates.append((_raw_domain_score(info, col), str(col), info, "column-name fallback"))

    if not candidates:
        return None, {}, "not found"

    candidates.sort(key=lambda item: (item[0], item[1]))
    _, col, info, reason = candidates[0]
    return col, info, reason


def _metadata_inverted(info: Mapping[str, Any]) -> bool:
    for key in ("invert", "inverted", "raw_inverted", "calibration_invert"):
        if isinstance(info.get(key), bool):
            return bool(info[key])
    calibration = info.get("calibration")
    if isinstance(calibration, Mapping) and isinstance(calibration.get("invert"), bool):
        return bool(calibration["invert"])
    return False


def _should_invert_raw(col: str | None, info: Mapping[str, Any], end: str) -> bool:
    if col is not None and str(col) in INVERT_RAW_COLUMNS:
        return True
    return bool(INVERT_RAW_BY_END.get(end, False)) or _metadata_inverted(info)


def _first_existing(df: pd.DataFrame, candidates: tuple[str, ...]) -> str | None:
    for col in candidates:
        if col in df.columns:
            return col
    return None


def _blank_series(index: pd.Index) -> pd.Series:
    return pd.Series([""] * len(index), index=index, dtype="object")


def _numeric_or_blank(df: pd.DataFrame, col: str | None, *, multiplier: float = 1.0) -> pd.Series:
    if col is None or col not in df.columns:
        return _blank_series(df.index)
    s = pd.to_numeric(df[col], errors="coerce") * float(multiplier)
    return s.where(np.isfinite(s), "")


def _sample_time_for_export(session: Mapping[str, Any]) -> pd.Series:
    df = session["df"]
    if TIME_FORMAT == "sample_count":
        return pd.Series(np.arange(len(df), dtype=np.uint64), index=df.index)
    if TIME_FORMAT == "elapsed_s":
        return pd.to_numeric(df["time_s"], errors="coerce")
    if TIME_FORMAT != "sample_count":
        raise ValueError(f"Unsupported TIME_FORMAT: {TIME_FORMAT!r}")
    raise ValueError(f"Unsupported TIME_FORMAT: {TIME_FORMAT!r}")


def _raw_counts_for_export(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce").round()
    return numeric.astype("Int64")



def build_data_syn_bike_export(session: Mapping[str, Any]) -> tuple[pd.DataFrame, dict[str, Any]]:
    df = session["df"]
    if "time_s" not in df.columns:
        raise ValueError("Session dataframe does not contain required time_s column")

    front_col, front_info, front_reason = _pick_raw_column(session, "front")
    rear_col, rear_info, rear_reason = _pick_raw_column(session, "rear")

    lat_col = _first_existing(df, LAT_COLS)
    lon_col = _first_existing(df, LON_COLS)
    speed_col = _first_existing(df, SPEED_COLS)

    front = _numeric_or_blank(df, front_col)
    rear = _numeric_or_blank(df, rear_col)

    front_inverted = _should_invert_raw(front_col, front_info, "front")
    rear_inverted = _should_invert_raw(rear_col, rear_info, "rear")
    if front_col is not None and front_inverted:
        front_num = pd.to_numeric(front, errors="coerce")
        front = (ADC_MAX_COUNT - front_num).where(np.isfinite(front_num), "")
    if rear_col is not None and rear_inverted:
        rear_num = pd.to_numeric(rear, errors="coerce")
        rear = (ADC_MAX_COUNT - rear_num).where(np.isfinite(rear_num), "")
    front = _raw_counts_for_export(front) if front_col is not None else front
    rear = _raw_counts_for_export(rear) if rear_col is not None else rear

    out = pd.DataFrame(
        {
            DATA_SYN_BIKE_COLUMNS["time"]: _sample_time_for_export(session),
            DATA_SYN_BIKE_COLUMNS["front_raw"]: front,
            DATA_SYN_BIKE_COLUMNS["rear_raw"]: rear,
            DATA_SYN_BIKE_COLUMNS["lat"]: _numeric_or_blank(df, lat_col),
            DATA_SYN_BIKE_COLUMNS["lon"]: _numeric_or_blank(df, lon_col),
            DATA_SYN_BIKE_COLUMNS["speed"]: _numeric_or_blank(df, speed_col, multiplier=SPEED_MULTIPLIER),
        }
    )

    meta = {
        "front_raw_col": front_col,
        "front_raw_reason": front_reason,
        "front_raw_inverted": front_inverted,
        "rear_raw_col": rear_col,
        "rear_raw_reason": rear_reason,
        "rear_raw_inverted": rear_inverted,
        "lat_col": lat_col,
        "lon_col": lon_col,
        "speed_col": speed_col,
        "time_format": TIME_FORMAT,
        "rows": len(out),
    }
    return out, meta

In [6]:
store = ArtifactStore(ARTIFACTS_DIR)
selected = selector["get_key_to_ref"]()
if not selected:
    raise ValueError("No sessions selected. Select one or more sessions first.")

written = []
for session_key, (run_id, session_id) in selected.items():
    artifacts = load_session_artifacts(store, run_id=run_id, session_id=session_id)
    session = {
        "session_id": session_id,
        "meta": artifacts.get("meta", {}),
        "df": artifacts["df"],
    }
    export_df, export_meta = build_data_syn_bike_export(session)

    filename = f"{_safe_filename_part(run_id)}__{_safe_filename_part(session_id)}__data_syn_bike.csv"
    out_path = OUTPUT_DIR / filename
    export_df.to_csv(out_path, index=False, header=False, na_rep="")
    written.append((out_path, export_meta))

for path, meta in written:
    print(f"Wrote {path} ({meta['rows']} rows)")
    print(
        "  raw columns: "
        f"front={meta['front_raw_col']} inverted={meta['front_raw_inverted']} ({meta['front_raw_reason']}), "
        f"rear={meta['rear_raw_col']} inverted={meta['rear_raw_inverted']} ({meta['rear_raw_reason']})"
    )
    print(
        f"  time format: {meta['time_format']}\n"
        "  gps columns: "
        f"lat={meta['lat_col']}, lon={meta['lon_col']}, speed={meta['speed_col']}"
    )

print(f"\nExport complete: {len(written)} file(s) in {OUTPUT_DIR.resolve()}")

Wrote exports\data_syn_bike\run_2026-04-27T22-19-55_AWST__2026-02-20_12-58-25__data_syn_bike.csv (231678 rows)
  raw columns: front=front_shock_raw_dom_suspension [counts] inverted=False (signal registry), rear=rear_shock_raw_dom_suspension [counts] inverted=False (signal registry)
  time format: sample_count
  gps columns: lat=gps_fit_position_latitude_dom_world [deg], lon=gps_fit_position_longitude_dom_world [deg], speed=gps_fit_enhanced_speed_dom_world [m/s]
Wrote exports\data_syn_bike\run_2026-04-27T22-19-55_AWST__2026-02-20_13-06-37__data_syn_bike.csv (32362 rows)
  raw columns: front=front_shock_raw_dom_suspension [counts] inverted=False (signal registry), rear=rear_shock_raw_dom_suspension [counts] inverted=False (signal registry)
  time format: sample_count
  gps columns: lat=gps_fit_position_latitude_dom_world [deg], lon=gps_fit_position_longitude_dom_world [deg], speed=gps_fit_enhanced_speed_dom_world [m/s]

Export complete: 2 file(s) in D:\Dev\BODAQS\analysis\exports\data_sy